# PlayHQ Season Competition: Team & Fixture Extraction

This notebook can be used to:

1. Find the seasons available under an organization (e.g., Coburg Association).
2. Find season ID from the season name, and vice-versa.
3. Extract all teams in a season competition (and save them into a CSV file).
4. Extract the fixtures of every team in the competition so far. If the games have been played, the result will be there too. Each team fixture is saved into a separate JSON file.

**NOTE:** this notebook is unrelated to TeamApps mobile app.

All the data is from [PlayHQ](http://playhq.com/) via its Public [API](https://support.playhq.com/hc/en-au/sections/4405422358297-PlayHQ-APIs). Detailed reference documentation for PlayHQ API can be found [here](https://docs.playhq.com/tech).

This notebook was first used to support the scheduler season capstone project in 2023.

**Contact:** Sebastian Sardina (sssardina@gmail.com)

In [14]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import json
import csv
import re
from pathlib import Path

import playhq as phq

## 1. Configuration and set-up

We first configure and set-up the application. This means reading configuration variables from a config file and setting the game day.

So, first of all, specify the following information:

1. Configuration file for the club and season.
2. Game dates interval to scrape.

In [ ]:
# Change this to import your club's own configuration
# from config_cba import *      # configuration file for CBA Rep league (private)
from config_bmc_w26 import *  # configuration file for BMC club in Winter 2026 season (private)

# Create phq_club object
phq_club = phq.PlayHQ(
    CLUB_NAME, ORG_ID, X_API_KEY, X_TENANT, TIMEZONE, tapp_team_name, tapp_game_name
)

print(f"Club name: {CLUB_NAME} (org. id: {ORG_ID})")
print("X-tenant:", X_TENANT, "x-api-key:", X_API_KEY)

print("Timezone:", TIMEZONE)
print("PlayHQ Club fixture:", PLAYHQ_SEASON_URL)


OUTPUT_ROOT_PATH = Path("fixtures-out")
if not OUTPUT_ROOT_PATH.exists():
    print("ERROR: Output path does not exist:", OUTPUT_ROOT_PATH)

Club name: Brunswick Magic Basketball Club (org. id: 8c4d5431-eaa5-4644-82ac-992abe224b88)
X-tenant: bv x-api-key: 3956576f-41f5-40c8-8f0f-a1bdb8dd03fe
Timezone: Australia/Melbourne
PlayHQ Club fixture: https://bit.ly/bmbc-w24


## 2. Get season id from season name

Find the season id from the season name, if we don't know it. It will also print the JSON received so one can manually check the seasons available for the organization.

To get the season ID, look for the `id` field in the JSON output. The season ID is a UUID string:

```json
            "competition": {
                "id": "81fe8d81-d087-4a63-b4fb-87bc85e9d8ac",
                "name": "Junior Domestic"
            },
            "createdAt": null,
            "endDate": "2026-09-19T13:59:59.000Z",
            "id": "e44fb190-fab0-491f-ae6d-8cb8d30ab24e",
            "name": "Winter 2026",
            "startDate": "2026-04-17T14:00:00.000Z",
            "status": "COMPLETED",
            "updatedAt": null
```

In this case the id is `e44fb190-fab0-491f-ae6d-8cb8d30ab24e`  for the season `Winter 2026` in the competition `Junior Domestic`.

In [ ]:
# get competition id
SEASON = "Winter 2023"
SEASON = "Summer 2023/24"
# SEASON = "Winter 2024"
SEASON = "Winter 2026"
SEASON = None  # use this if you don't know and just want to list them all

season_id = None
competition_id = None
for data_json in phq_club.get_json(f"organisations/{ORG_ID}/seasons"):
    print(json.dumps(data_json, sort_keys=True, indent=4))
    for x in data_json["data"]:
        if SEASON is None:
            continue
        if x["name"] == SEASON:
            season_id = x["id"]
            # competition_id = x['competition']['id']
            print(f"Seasons *{SEASON}* found with id: {season_id}")

{
    "data": [
        {
            "association": {
                "id": "e18085db-c6de-512f-b78b-53d253c65b32",
                "logo": {
                    "sizes": [
                        {
                            "dimensions": {
                                "height": 32,
                                "width": 32
                            },
                            "url": "https://res.cloudinary.com/playhq/image/upload/h_32,w_32/v1/production/bv/e18085db-c6de-512f-b78b-53d253c65b32/1645069189211/logo.jpg?_a=BAMAMifg0"
                        },
                        {
                            "dimensions": {
                                "height": 48,
                                "width": 48
                            },
                            "url": "https://res.cloudinary.com/playhq/image/upload/h_48,w_48/v1/production/bv/e18085db-c6de-512f-b78b-53d253c65b32/1645069189211/logo.jpg?_a=BAMAMifg0"
                        },
                      

## 3. Set the season id to scrape

First set the season id we want to scrape. We can get it from the name using the above cell, but it has to be set manually here. 

Once set, find the season name and set the output path where all output files (including the fixture per team) will be saved.

In [10]:
season_id = "a94981b4-75b7-429f-9005-915182ab6153" # Victorian Junior Domestic - Summer 2022/23
season_id = "cdbe3065-2a32-4c6d-8771-f8fae3fa7611" # Victorian Junior Domestic - Winter 2023
season_id = "2485b3f7-55de-4b88-a3b6-527c36717fee" # Victorian Junior Domestic - Summer 2023/24
season_id = "e44fb190-fab0-491f-ae6d-8cb8d30ab24e" # Victorian Junior Domestic - Winter 2026

# season_id = "78824ad1-0ca6-46c5-9440-dbe20c948b2f"  # VJBL 2023
# season_id = "8d724c0a-3a0f-4da7-92f8-2e2053a6707b"  # National Junior 2023

season_name = None
for data_json in phq_club.get_json(f"organisations/{ORG_ID}/seasons"):
    # print(json.dumps(data_json, sort_keys=True, indent=4))

    for x in data_json['data']:
        if x['id'] == season_id:
            season_name = x['name']
            break


print(f'Found season name for *{season_id}*: {season_name}')

OUTPUT_PATH = OUTPUT_ROOT_PATH / season_name.lower().replace(' ', '_').replace('/', '-')

if not OUTPUT_PATH.exists():
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f'Output PATH to store fixtures ready: {OUTPUT_PATH}')


Found season name for *e44fb190-fab0-491f-ae6d-8cb8d30ab24e*: Winter 2026
Output PATH to store fixtures ready: fixtures-out/winter_2026


## 4. Extract participating teams

Get all the teams participating in the competition. Save the list to file `teams.csv`

In [12]:
teams = []
for data_json in phq_club.get_json(f"seasons/{season_id}/teams"):
    print(data_json)
    if data_json['data'][0]['club'] is None:
        break
    for t in data_json['data']:
        # print(t)
        team = {}

        team['id'] = t['id']
        team['club'] = t['club']['name'] if t['club'] is not None else "Unknown"
        if t['grade'] is None:
            continue    # skip teams without grade
        team['grade'] = t['grade']['name']
        team['name'] = t['name']

        teams.append(team)
    # print(json.dumps(data_json, sort_keys=True, indent=4))

print("Number of teams extracted:", len(teams))
print(teams[10])


fieldnames = ["id", "club", "grade", "name"]
with open(OUTPUT_PATH / "teams.csv", 'w', encoding='UTF8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(teams)


{'data': [{'id': '47b53b38-84df-4c91-9842-9df82ed36263', 'name': 'Jets Open Mixed EOI', 'createdAt': '2026-03-16T11:18:00.000Z', 'updatedAt': '2026-03-16T11:18:00.000Z', 'club': {'id': '3e2ecab8-1643-4ba7-b557-13bd5503a7b1', 'name': 'Northcote Jets Basketball Club Inc.', 'url': 'https://www.playhq.com/basketball-victoria/org/northcote-jets-basketball-club-inc/3e2ecab8', 'logo': {'sizes': [{'url': 'https://res.cloudinary.com/playhq/image/upload/h_32,w_32/v1/production/bv/3e2ecab8-1643-4ba7-b557-13bd5503a7b1/1776390835394/logo.png?_a=BAMAMifg0', 'dimensions': {'width': 32, 'height': 32}}, {'url': 'https://res.cloudinary.com/playhq/image/upload/h_48,w_48/v1/production/bv/3e2ecab8-1643-4ba7-b557-13bd5503a7b1/1776390835394/logo.png?_a=BAMAMifg0', 'dimensions': {'width': 48, 'height': 48}}, {'url': 'https://res.cloudinary.com/playhq/image/upload/h_128,w_128/v1/production/bv/3e2ecab8-1643-4ba7-b557-13bd5503a7b1/1776390835394/logo.png?_a=BAMAMifg0', 'dimensions': {'width': 128, 'height': 128}}

## 5. Extract fixture for each team

Now extract the fixture so far of every team in the competition. Store each team fixture in a JSON file. Note that games will appear in two files always: for the home and away teams.

In [15]:
# team_id = "139ecf7c-55e5-411f-bbce-4bceb321f342"
# team_id="b55cb46c-8415-4b6e-9ac5-ad38b02e5b35"

for team in teams:
    team_id = team['id']
    standard_name = re.sub(r'[\W_]', "_", team['name'])
    file_name = OUTPUT_PATH / (standard_name + ".json")

    if file_name.exists():
        continue
    print("Processing team:", team['name'])

    games = {}
    games["data"] = []

    for data_json in phq_club.get_json(f"teams/{team_id}/fixture"):
        # if data_json['data'][0]['club'] is None:
        #     break
        games["data"] = games["data"] + data_json['data']

    with open(file_name, "w") as f:
        f.write(json.dumps(games, sort_keys=True, indent=4))


print("Number of teams fixture extracted:", len(teams))
# print(teams[10])

Processing team: Jets U10 Boys Blue
Processing team: Jets U10 Boys Orange
Processing team: Jets U10 Boys Red
Processing team: Jets U10 Boys Yellow
Processing team: Jets U10 Girls Blue
Processing team: Jets U10 Girls Yellow
Processing team: Jets U12 Boys Black
Processing team: Jets U12 Boys Blue
Processing team: Jets U12 Boys Red
Processing team: Jets U12 Boys Yellow
Processing team: Jets U12 Girls Yellow
Processing team: Jets U12 Mixed Orange
Processing team: Jets U14 Boys Blue
Processing team: Jets U14 Boys Red
Processing team: Jets U14 Boys Yellow
Processing team: Jets U14 Girls Blue
Processing team: Jets U14 Girls Yellow
Processing team: Jets U16 Boys Yellow
Processing team: Jets U16 Girls Yellow
Processing team: Jets U18 Boys Blue
Processing team: Jets U18 Boys Yellow
Processing team: Jets U18 Girls Blue
Processing team: Jets U20 Boys Blue
Processing team: Jets U20 Girls Yellow
Processing team: Magic U10 Boys Diamond
Processing team: Magic U10 Boys Gold
Processing team: Magic U10 B

To print the fixture of a particular team:

In [17]:
team_id = "fa2a5f4a-dd9d-49a5-afde-2bc3d2417efa"

for data_json in phq_club.get_json(f"teams/{team_id}/fixture"):
    print(json.dumps(data_json, sort_keys=True, indent=4))
    # print(data_json)

{
    "data": [
        {
            "competitors": [
                {
                    "id": "7b0a1b72-455b-46a0-a6b3-6439e132143c",
                    "isHomeTeam": true,
                    "name": "Magic U16 Boys Orange",
                    "outcome": "LOST",
                    "scoreSubTotal": [
                        {
                            "type": "3_POINT_SCORE",
                            "value": 3
                        },
                        {
                            "type": "1_POINT_SCORE",
                            "value": 2
                        },
                        {
                            "type": "2_POINT_SCORE",
                            "value": 9
                        },
                        {
                            "type": "TOTAL_FOULS",
                            "value": 0
                        }
                    ],
                    "scoreTotal": 29
                },
                {
                